In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import textwrap
import time

import langchain

from langchain.chains import RetrievalQA

import transformers
import torch


print('LangChain:', langchain.__version__)
print('Transformers', transformers.__version__)

LangChain: 0.2.7
Transformers 4.42.3


In [2]:
from transformers import AutoTokenizer, pipeline, logging
import argparse
from langchain import LLMChain
from langchain.prompts.prompt import PromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, AutoConfig
from langchain.document_loaders import DirectoryLoader
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceInstructEmbeddings
from InstructorEmbedding import INSTRUCTOR

In [3]:
class PVABot:
    # LLMs
    model_name = 'Llama-3-SauerkrautLM-8b-Instruct' 
    temperature = 0.1,
    top_p = 0.95,
    repetition_penalty = 1.15    
    per_device_train_batch_size = 4,
    # splitting
    split_chunk_size = 2000
    split_overlap = 0
    
    num_train_epochs = 50

    # similar passages
    k = 10
    
    path = os.getcwd()
    print("Current_Directory:", path)

    abs = os.path.abspath(os.path.join(path, os.pardir))

    print("Absolute_Path:", abs)

    embeddings_model_repo = os.path.join(abs, 'input/models/sentence-t5-xxl/')
    print("Embedding_Model_Path:", embeddings_model_repo)

    PDFs_path = os.path.join(abs, 'input/wim/')
    print("PDF_Directory:", PDFs_path)
    FAISS_path = os.path.join(abs, 'input/db_faiss')
    print("FAISS_Data:", FAISS_path)

    model_path = os.path.join(abs, 'input/models/Llama-3-SauerkrautLM-8b-Instruct')
    print("Model_Path:", model_path)

Current_Directory: /home/02u21y0k/Development/WIM_Auskunft_Chatbot/notebooks
Absolute_Path: /home/02u21y0k/Development/WIM_Auskunft_Chatbot
Embedding_Model_Path: /home/02u21y0k/Development/WIM_Auskunft_Chatbot/input/models/sentence-t5-xxl/
PDF_Directory: /home/02u21y0k/Development/WIM_Auskunft_Chatbot/input/db_faiss/wim/
FAISS_Data: /home/02u21y0k/Development/WIM_Auskunft_Chatbot/input/db_faiss
Model_Path: /home/02u21y0k/Development/WIM_Auskunft_Chatbot/input/models/Llama-3-SauerkrautLM-8b-Instruct


In [4]:
def get_model(model = PVABot.model_name):

    path = os.getcwd()
    print("Current Directory", path)

    abs = os.path.abspath(os.path.join(path, os.pardir))

    PDFs_path = os.path.join(abs, 'input/wim')
    print("PDF Directory", PDFs_path)
    FAISS_path = os.path.join(abs, 'input/db_faiss/')
    model_path = os.path.join(abs, 'input/models/Llama-3-SauerkrautLM-8b-Instruct')
    print(model_path)

    print('\nDownloading model: ', model, '\n\n')

    if model == 'Llama-3-SauerkrautLM-8b-Instruct':
        model_repo = model_path

        tokenizer = AutoTokenizer.from_pretrained(model_repo, use_fast=True, trust_remote_code=True) 

        #'device': 'cuda:0', 'max_memory': None, 'quantize_config': None

        #model = ORTModelForQuestionAnswering.from_pretrained(
        #    model_repo,
        #    load_in_8bit=False,
        #    device_map='cpu', 
        #    low_cpu_mem_usage=True,
        #    trust_remote_code=True,
        #    use_safetensors=True,
        #    attn_implementation="flash_attention_2",
        #)

        model = AutoModelForCausalLM.from_pretrained(
            model_repo,
            load_in_8bit=True,
            torch_dtype=torch.float16,
            device_map='cuda', 
            low_cpu_mem_usage=True,
            trust_remote_code=True,
            use_safetensors=True,
            #attn_implementation="flash_attention_2",
        )
        
        max_len = 10000

    else:
        print("Not implemented model (tokenizer and backbone)")

    return tokenizer, model, max_len


In [5]:
print(transformers.__version__)

4.42.3


In [6]:
import torch
torch.cuda.is_available()

True

In [7]:
%%time

tokenizer, model, max_len = get_model(model = PVABot.model_name)

Current Directory /home/02u21y0k/Development/WIM_Auskunft_Chatbot/notebooks
PDF Directory /home/02u21y0k/Development/WIM_Auskunft_Chatbot/input/db_faiss/wim
/home/02u21y0k/Development/WIM_Auskunft_Chatbot/input/models/Llama-3-SauerkrautLM-8b-Instruct





Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

CPU times: user 24.8 s, sys: 4.18 s, total: 29 s
Wall time: 4.5 s


In [8]:
pipe = pipeline(
    task = "text-generation",
    model = model,
    tokenizer = tokenizer,
    pad_token_id = tokenizer.eos_token_id,
    max_length = max_len,
    temperature = PVABot.temperature,
    top_p = PVABot.top_p,
    repetition_penalty = PVABot.repetition_penalty,
)

llm = HuggingFacePipeline(pipeline = pipe)

/app/hsthome/02u21y0k/.local/lib/python3.9/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 0.3. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFacePipeline`.
  warn_deprecated(


In [ ]:
#from vllm import LLM
#llm = LLM(model_path, tensor_parallel_size=2)

In [9]:
llm

HuggingFacePipeline(pipeline=<transformers.pipelines.text_generation.TextGenerationPipeline object at 0x7f9b4b023730>)

In [ ]:
#prompt = "Allways answer in German. Antworten Sie immer auf Deutsch. Antworte immer mit dem Wissen aus word,translation,meaning. Was heißt a Spuckerl entfernt" 
#response = llm.generate([prompt])
#print(response.generations[0][0].text)

In [10]:
PVABot.model_name

'Llama-3-SauerkrautLM-8b-Instruct'

In [11]:
%%time

loader = DirectoryLoader(
    PVABot.PDFs_path,
    glob="./*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True,
    use_multithreading=True,
)

documents = loader.load()

100%|██████████| 4/4 [00:02<00:00,  1.56it/s]

CPU times: user 2.56 s, sys: 12.1 ms, total: 2.57 s
Wall time: 2.56 s


----------------------------------------------------------------------------------------

## SFT Training

In [ ]:
path = os.getcwd()
print("Current_Directory:", path)

abs = os.path.abspath(os.path.join(path, os.pardir))

#abs = os.path.abspath(os.path.join(path, os.pardir))
# Model from Hugging Face hub
base_model = os.path.join(abs, 'input/models/Llama-3-SauerkrautLM-8b-Instruct')

# Fine-tuned model
new_model = os.path.join(abs, 'outputs/Llama-3-SauerkrautLM-8b-Instruct_SFT')

In [ ]:
print(base_model)

In [ ]:
from datasets import load_dataset

abs = os.path.abspath(os.path.join(path, os.pardir))
print(abs)

dataset_path = os.path.join(abs, 'input/datasets/oewb/oewb_for_sft.csv')
dataset = load_dataset("csv", data_files=dataset_path, split="train")

In [ ]:
dataset

from datasets import load_dataset

abs = os.path.abspath(os.path.join(path, os.pardir))

dataset_path = os.path.join(abs, 'notebooks/oesterreichisch_clean.csv')
dataset = load_dataset("text", data_files=dataset_path, split="train", encoding="iso-8859-1")
#dataset = load_dataset("csv", data_files=dataset_path, split="train", encoding="iso-8859-1")

In [ ]:
prompt = "Allways answer in German. Antworten Sie immer auf Deutsch. Antworte immer mit dem Wissen aus word,translation,meaning" 
response = llm.generate([prompt])
print(response.generations[0][0].text)

In [ ]:
# Preprocess the data
def preprocess_function(examples):
    inputs = [example['deutsch'] for example in examples["train"]]
    targets = [example['oesterreichisch'] for example in examples["train"]]
    model_inputs = tokenizer(inputs, text_target=targets, max_length=128, truncation=True)
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)


In [ ]:

from peft import LoraConfig, get_peft_model
from transformers import (
    TrainingArguments,
)

peft_params = LoraConfig(
    #r=8, 
    #lora_alpha=32, 
    #lora_dropout=0.05, 
    lora_alpha=16,
    lora_dropout=0.5,
    r=16,
    bias="none",
    task_type="CAUSAL_LM",
    #target_modules=["q_proj", "v_proj"]
    target_modules=['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj']
)

In [ ]:
training_params = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4, 
    #4 epochen sollten reichen
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=1000,
    logging_steps=10,
    learning_rate=5e-6,
    weight_decay=0.0,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=200,
    warmup_steps=100,
    warmup_ratio=0.1,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="tensorboard"
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_params,
    dataset_text_field="text",
    max_seq_length=None,
    tokenizer=tokenizer,
    args=training_params,
    packing=False,
)

In [ ]:
trainer.model.save_pretrained(new_model)
trainer.tokenizer.save_pretrained(new_model)

In [ ]:
from transformers import AutoModelForCausalLM

from peft import PeftModel

abs = os.path.abspath(os.path.join(path, os.pardir))
model_path = os.path.join(abs, 'input/models/Llama-3-SauerkrautLM-8b-Instruct')
adapter_path = os.path.join(abs, 'outputs/Llama-3-SauerkrautLM-8b-Instruct_SFT')

base_model = AutoModelForCausalLM.from_pretrained(model_path)

peft_model_id = adapter_path

model = PeftModel.from_pretrained(base_model, peft_model_id)

merged_model = model.merge_and_unload()

In [ ]:
model_test = os.path.join(abs, 'outputs/SauerkrautLM-8b-Instruct_SFT_trained')

merged_model.save_pretrained(model_test)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/runs

-----------------------------------------------------------

## DPO Training:

In [ ]:
abs = os.path.abspath(os.path.join(path, os.pardir))
print(abs)
# Model from Hugging Face hub
base_model = os.path.join(abs, 'input/models/Llama-3-SauerkrautLM-8b-Instruct_tirol_DPO')

# Fine-tuned model
new_model = os.path.join(abs, 'outputs/Llama-3-SauerkrautLM-8b-Instruct_tirol_wien_DPO')

In [ ]:
def dpo_format(example):
    # Format instruction
    question = example['meaning']

    # Format chosen answer
    chosen = example['word'] 

    # Format rejected answer
    rejected = example['translation']

    return {
        "prompt": str(question),
        "chosen": chosen,
        "rejected": str(rejected),
    }

In [ ]:
from datasets import load_dataset

abs = os.path.abspath(os.path.join(path, os.pardir))
print(abs)

dataset_path = os.path.join(abs, 'input/datasets/oewb/oewb_for_dpo.csv')
dataset = load_dataset("csv", data_files=dataset_path, split="train", encoding="iso-8859-1")
original_columns = dataset.column_names

In [ ]:
for column in original_columns:
    print(f"Data in column '{column}':")
    print(dataset[column])
    print()

In [ ]:
dataset[1]
print(dataset)

In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

In [ ]:
# Format dataset
dataset = dataset.map(
    dpo_format,
    remove_columns=original_columns
)

# Print sample
dataset[1]

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj']
)

In [ ]:
# Model to fine-tune
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    load_in_4bit=True
)
model.config.use_cache = False

In [ ]:
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
# Training arguments
training_args = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    max_steps=200,
    save_strategy="no",
    logging_steps=1,
    output_dir=new_model,
    optim="paged_adamw_32bit",
    warmup_steps=100,
    bf16=True,
)

In [ ]:
print(type(dataset))

In [ ]:
from trl import DPOTrainer
# Create DPO trainer
dpo_trainer = DPOTrainer(
    model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    peft_config=peft_config,
    beta=0.1,
    max_prompt_length=1024,
    max_length=1536,
)

In [ ]:
# Fine-tune model with DPO
dpo_trainer.train()

In [ ]:
dpo_trainer.model.save_pretrained("kaiserschmarrnLM_tirol_wien_dpo")
tokenizer.save_pretrained("kaiserschmarrnLM_tirol_wien_dpo")

In [ ]:
from transformers import AutoModelForCausalLM

from peft import PeftModel

import os

path = os.getcwd()

print(path)

#/home/02u21y0k/PVA_Chatbot/notebooks/tirolerisch_adapter

abs = os.path.abspath(os.path.join(path, os.pardir))
print(abs)
model_path = os.path.join(abs, 'input/models/Llama-3-SauerkrautLM-8b-Instruct_tirol_DPO') 
adapter_path = os.path.join(abs, 'input/models/kaiserschmarrnLM_tirol_wien_dpo')

base_model = AutoModelForCausalLM.from_pretrained(model_path)

peft_model_id = adapter_path

model = PeftModel.from_pretrained(base_model, peft_model_id)

merged_model = model.merge_and_unload()

## Splitter

In [ ]:
merged_model.save_pretrained(new_model)
tokenizer.save_pretrained(new_model)

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
new_model = os.path.join(abs, 'input/Llama-3-SauerkrautLM-8b-Instruct_DPO')

In [ ]:
merged_model.push_to_hub('', private=True)
tokenizer.push_to_hub('', private=True)

In [ ]:
# format prompt
message=[
    {"role":"system", "content":"Write in GERMAN"},
    {"role":"user","content":"?"}
]

#tokenizer=AutoTokenizer.from_pretrained(os.getenv("WANDB_NAME"))
prompt=tokenizer.apply_chat_template(message, add_generation_prompt=True, tokenize=False)

pipe=transformers.pipeline(
    "text-generation",
    model,
    #model=os.getenv("WANDB_NAME"),
    tokenizer=tokenizer
)

# generate text
sequences=pipe(
    prompt,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    num_return_sequences=1,
    max_length=200,
)

sequences[0]['generated_text']

-------------------------------------------------------------------

In [12]:
print(f'We have {len(documents)} pages in total')

We have 62 pages in total


In [13]:
documents[8].page_content

'E. Sonstige Leistungen\n1. Pflegegeld nach dem Bundespflegegeldgesetz\nStufe 1  ..............................................................................................................................................€192,00\nStufe 2  ..............................................................................................................................................€354,00\nStufe 3  ..............................................................................................................................................€551,60\nStufe 4  ..............................................................................................................................................€827,10\nStufe 5  ..............................................................................................................................................€1.123,50\nStufe 6  .....................................................................................................................

In [14]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = PVABot.split_chunk_size,
    chunk_overlap = PVABot.split_overlap
)

texts = text_splitter.split_documents(documents)

print(f'We have created {len(texts)} chunks from {len(documents)} pages')

We have created 80 chunks from 62 pages


In [16]:
%%time
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Qdrant

### download embeddings model
embeddings = HuggingFaceInstructEmbeddings(
    model_name = PVABot.embeddings_model_repo,
    model_kwargs = {"device": "cuda"},
)

### load vector DB embeddings
#vectordb = FAISS.load_local(
#    PVABot.Embeddings_path,
#    embeddings,
#    allow_dangerous_deserialization=True,
#)

qdrant = Qdrant.from_documents(
    documents,
    embeddings,
    location=":memory:",  # Local mode with in-memory storage only
    collection_name="reranker",
)

load INSTRUCTOR_Transformer
max_seq_length  512
CPU times: user 53min 39s, sys: 9min 59s, total: 1h 3min 39s
Wall time: 4min 9s


In [ ]:
qdrant.similarity_search('Regelpensionsalter')

In [ ]:
qdrant.similarity_search('In welchen Monaten gebührt bei den Pensionen eine Sonderzahlung?')

In [ ]:
prompt_template = """
Allways answer in German.
Antworten Sie immer auf Deutsch.
Antworte Sie immer für Österreich.
Pensionsversicherung in Österreich.
Antworte Sie immer mit der österreichischen Gesetzgebung.
Versuchen Sie eine Antwort aus den vorhandenen Daten zu finden. 
Wenn Sie es nicht wissen, sagen Sie einfach, dass Sie es nicht wissen und lügen Sie nicht.
Antworten Sie in der gleichen Sprache, in der die Frage gestellt wurde.
Verwenden Sie zur Beantwortung der abschließenden Frage nur die folgenden Kontextteile.

{context}

Question: {question} Antworten Sie immer auf Deutsch. Antworte Sie immer mit der österreichischen Gesetzgebung. Pensionsversicherung in Österreich.
Antwort:"""


PROMPT = PromptTemplate(
    template = prompt_template, 
    input_variables = ["context", "question"]
)

In [20]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder


path = os.getcwd()
print("Current_Directory:", path)

#retriever = vectordb.as_retriever(search_kwargs = {"k": PVABot.k, "search_type" : "similarity"})
abs = os.path.abspath(os.path.join(path, os.pardir))
print(abs)

retriever = qdrant.as_retriever(search_kwargs = {"k": PVABot.k})
#retriever = qdrant.as_retriever(search_kwargs = {"k": None})

#model = HuggingFaceCrossEncoder(model_name="/app/hsthome/02u21y0k/PVA_Chatbot/notebooks/input/bge_reranker_skillfit")
model = HuggingFaceCrossEncoder(model_name=os.path.join(abs, 'input/models/bge_reranker_skillfit/'))

compressor = CrossEncoderReranker(model=model, top_n=10)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

Current_Directory: /home/02u21y0k/Development/WIM_Auskunft_Chatbot/notebooks
/home/02u21y0k/Development/WIM_Auskunft_Chatbot


In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm = llm,
    chain_type = "stuff",
    retriever = compression_retriever, 
    chain_type_kwargs = {"prompt": PROMPT},
    return_source_documents = True,
    verbose = False
)

In [ ]:
# Define a dictionary to store conversation chains for each user
conversation_chains = {}
from uuid import uuid4

history = {}
session_id = str(uuid4())

def handle_user_input(session_id, user_input):
    # If the user doesn't have a conversation chain, create one
    if session_id not in conversation_chains:
        prompt = PromptTemplate(
            input_variables=["history", "input"],
            template="You are a helpful AI assistant. The user's previous messages are: {history}. The user's current message is: {input}.",
        )
        conversation_chains[session_id] = ConversationChain(prompt=prompt, llm=llm, verbose=True)

    # Get the user's conversation chain
    conversation = conversation_chains[session_id]

    # Generate a response
    response = conversation.predict(input=user_input)

    return response

In [ ]:
# Define a custom chain that includes handle_user_input
from langchain.chains import ConversationalRetrievalChain

class CustomConversationalRetrievalChain(ConversationalRetrievalChain):
    def _call(self, inputs):
        question = inputs["question"]
        session_id = inputs["session_id"]

        # Get the chat history for the session
        history = self.memory.load_memory_variables({})["history"]

        # Generate a response using handle_user_input
        response = handle_user_input(session_id, question)

        # Add the user's question and the chatbot's response to the chat history
        history.append((question, response))

        # Use the retrieval chain to get relevant documents
        docs = self.retriever.get_relevant_documents(question)

        # Generate an answer using the retrieved documents and the chat history
        answer = self.combine_documents_chain.run(input_documents=docs, question=question, history=history)

        # Update the chat history in the memory
        self.memory.save_context({"input": question}, {"output": answer})

        return {"answer": answer, "source_documents": docs}

In [ ]:
qa_chain = CustomConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=compression_retriever,
    combine_docs_chain_kwargs={"prompt": PROMPT},
    verbose=False
)

In [ ]:
print(type(qa_chain))

In [ ]:
### testing MMR search
question = "In welchen Monaten gebührt bei den Pensionen eine Sonderzahlung?"
qdrant.max_marginal_relevance_search(question, k = PVABot.k)

In [ ]:
### testing similarity search
question = "Was passiert bei einem späten Pensionsantritt?"
qdrant.similarity_search(question, k = PVABot.k)

In [ ]:
def wrap_text_preserve_newlines(text, width=700):
    # Split the input text into lines based on newline characters
    lines = text.split('\n')

    # Wrap each line individually
    wrapped_lines = [textwrap.fill(line, width=width) for line in lines]

    # Join the wrapped lines back together using newline characters
    wrapped_text = '\n'.join(wrapped_lines)

    return wrapped_text


def process_llm_response(llm_response):
    ans = wrap_text_preserve_newlines(llm_response['result'])
    
    sources_used = ' \n'.join(
        [
            source.metadata['source'].split('/')[-1][:-4] + ' - page: ' + str(source.metadata['page'])
            for source in llm_response['source_documents']
        ]
    )
    
    ans = ans #+ '\n\nSources: \n' + sources_used
    return ans

In [ ]:
def process_llm_response(llm_response):
    result = llm_response['result']
    source_documents = llm_response['source_documents']

    # Wrap the text while preserving newlines
    answer = wrap_text_preserve_newlines(result)

    # Find the index of the line containing "Question"
    lines = answer.split('\n')
    question_index = next((i for i, line in enumerate(lines) if "Question" in line), None)

    # Get the relevant lines
    if question_index is not None:
        answer = '\n'.join(lines[question_index+1:])
    else:
        answer = '\n'.join(lines)

    # Get the source information
    sources_used = '\n'.join(
        f"{source.metadata['source'].split('/')[-1][:-4]} - page: {source.metadata['page']}"
        for source in source_documents
    )

    # Append the source information to the answer
    answer += '\n\nSources:\n' + sources_used

    return answer

In [ ]:
# Define a dictionary to store conversation chains for each user
conversation_chains = {}
from uuid import uuid4

history = {}
session_id = str(uuid4())

def handle_user_input(session_id, user_input):
    # If the user doesn't have a conversation chain, create one
    if session_id not in conversation_chains:
        prompt = PromptTemplate(
            input_variables=["history", "input"],
            template="You are a helpful AI assistant. The user's previous messages are: {history}. The user's current message is: {input}.",
        )
        conversation_chains[session_id] = ConversationChain(prompt=prompt, llm=llm, verbose=True)

    # Get the user's conversation chain
    conversation = conversation_chains[session_id]

    # Generate a response
    response = conversation.predict(input=user_input)

    return response

In [ ]:
def llm_ans(query):
    start = time.time()
    llm_response = qa_chain(query)
    ans = process_llm_response(llm_response)
    end = time.time()

    time_elapsed = int(round(end - start, 0))
    time_elapsed_str = f'\n\nTime elapsed: {time_elapsed} s'
    return ans + time_elapsed_str

In [ ]:
PVABot.model_name

In [ ]:
model

In [22]:
import panel as pn

path = os.getcwd()
print("Current_Directory:", path)

pn.extension(design="material")

def predict(message, history, instance):

    output = str(llm_ans(message)).replace("\n", "<br/>")
    return output

chat_feed = pn.chat.ChatFeed()

chat_interface = pn.chat.ChatInterface(
    callback=predict,
    user="TKS-Mitarbeiter",
    callback_user="PVAlbert",
    message_params = dict( default_avatars={ "TKS-Mitarbeiter": "🧑", "PVAlbert": "/home/02u21y0k/Development/WIM_Auskunft_Chatbot/logo/pvalbertchatbubble.svg" }), 
)

chat_interface.send(
    "Hallo, ich bin PVAlbert, Ihr digitaler Assistent! Ich bin rund um die Uhr für Sie da, stellen Sie mir einfach Ihre Fragen. Mein Wissen über die pensionsversicherungsspezifischen Themen kombiniere ich mit Mistral-Technologie. Los geht's, ich freue mich Sie unterstützen zu dürfen!",
    user="PVAlbert",
    respond=False,
)

chat_interface.send(
    "Was darf am Stichtag nicht vorliegen, damit eine Korridorpension anfallen kann?",
)

pn.Column(
    chat_interface,
).servable()

pn.serve(chat_interface, port=8009)

Current_Directory: /home/02u21y0k/Development/WIM_Auskunft_Chatbot/notebooks


Launching server at http://localhost:8009


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
